In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-12b-it",
    max_seq_length = 16384,
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    load_in_16bit = True,
    full_finetuning = False, # [NEW!] We have full finetuning now!
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 128,           # Larger = higher accuracy, but might overfit
    lora_alpha = 256,  # Recommended alpha == r at least
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    random_state = 3407,
)

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("reranking_dataset_doris_mae")

In [ ]:
dataset[100]

In [ ]:
import pandas as pd

corpus_df = pd.read_json("splits/Corpus.json")

In [ ]:
corpus_df.head()

In [ ]:
len(corpus_df)


In [ ]:
print("Creating abstract_id -> text lookup dictionary...")
abstract_lookup = {}
for _, row in corpus_df.iterrows():
    title = row['title'].replace("_", " ")
    text = f"Title: {title}\nAbstract: {row['original_abstract']}"
    abstract_lookup[int(row['abstract_id'])] = text

print(f"Created lookup for {len(abstract_lookup)} abstracts.")

In [ ]:
abstract_lookup[0]

In [ ]:
import random
import string

# Document identifiers for FIRST (using letters A-Z, then combinations)
def get_doc_ids(n):
    """Generate document identifiers for n documents"""
    ids = list(string.ascii_uppercase)  # A-Z (26)
    if n > 26:
        # Add AA, AB, AC... if needed
        for c1 in string.ascii_uppercase:
            for c2 in string.ascii_uppercase:
                ids.append(c1 + c2)
                if len(ids) >= n:
                    return ids[:n]
    return ids[:n]

# FIRST prompt template
FIRST_SYSTEM_PROMPT = """You are a scientific literature search assistant. Given a query and a list of candidate documents, identify the most relevant document by outputting only its identifier letter."""

def create_first_prompt(query, doc_ids_and_texts):
    """Create FIRST-style prompt for listwise reranking"""
    docs_text = "\n\n".join([
        f"[{doc_id}] {text}" 
        for doc_id, text in doc_ids_and_texts
    ])
    
    prompt = f"""Query: {query}

Candidate Documents:
{docs_text}

Which document is most relevant to the query? Output only the document identifier."""
    
    return prompt

In [ ]:
len(dataset)

In [ ]:
len(dataset[0]['candidate_pool'])

In [ ]:
bad_items = [
    (i, item)
    for i, item in enumerate(dataset)
    if not any(score > 0 for score in item["candidate_scores"])
]

print(f"Number of queries with zero positives: {len(bad_items)}\n")

for i, item in bad_items:
    print(f"Index: {i}")
    print(f"Query: {item.get('query_text', '<no query_text>')}")
    print(f"Candidate scores: {item['candidate_scores']}")
    print("-" * 80)


In [ ]:
# Cell: Updated dataset creation with relevance scores and token IDs
import json
import random

def create_first_dataset(raw_data_path, abstract_lookup, tokenizer, 
                            num_candidates=50, samples_per_query=1):
    """
    Create FIRST training dataset with:
    - Document identifier token IDs
    - Relevance scores for each document
    - Properly formatted prompts
    """
    training_examples = []
    doc_id_letters = get_doc_ids(num_candidates)
    
    # Pre-compute token IDs for document identifiers
    doc_letter_to_token = {}
    for letter in doc_id_letters:
        tokens = tokenizer.encode(letter, add_special_tokens=False)
        doc_letter_to_token[letter] = tokens[0]  # First token
    
    with open(raw_data_path, 'r') as f:
        for line in f:
            example = json.loads(line)
            query = example['query_text']
            candidate_pool = example['candidate_pool']
            candidate_scores = example['candidate_scores']
            
            scored_candidates = list(zip(candidate_pool, candidate_scores))
            
            # Need at least some relevant documents
            has_relevant = any(s > 0 for _, s in scored_candidates)
            if not has_relevant:
                continue
            
            for _ in range(samples_per_query):
                # Sample candidates ensuring diversity
                if len(scored_candidates) >= num_candidates:
                    selected = random.sample(scored_candidates, num_candidates)
                else:
                    selected = scored_candidates.copy()
                
                # Shuffle to randomize positions
                random.shuffle(selected)
                
                # Build prompt and collect metadata
                doc_ids = get_doc_ids(len(selected))
                doc_ids_and_texts = []
                relevance_scores = []
                doc_token_ids = []
                
                for i, (cid, score) in enumerate(selected):
                    doc_id = doc_ids[i]
                    text = abstract_lookup.get(cid, f"Document {cid}")
                    doc_ids_and_texts.append((doc_id, text))
                    relevance_scores.append(float(score))
                    doc_token_ids.append(doc_letter_to_token[doc_id])
                
                # Find best document for the target (used in prompt, but loss uses all)
                best_idx = relevance_scores.index(max(relevance_scores))
                target_id = doc_ids[best_idx]
                
                prompt = create_first_prompt(query, doc_ids_and_texts)
                
                training_examples.append({
                    'prompt': prompt,
                    'target': target_id,
                    'query': query,
                    'relevance_scores': relevance_scores,
                    'doc_token_ids': doc_token_ids,
                    'num_candidates': len(selected)
                })
    
    return training_examples

# Create the updated dataset
first_examples = create_first_dataset(
    "reranking_dataset_doris_mae.jsonl",
    abstract_lookup,
    tokenizer.tokenizer,
    num_candidates=50,
    samples_per_query=3
)

print(f"Created {len(first_examples)} FIRST training examples")
print(f"\nExample relevance scores: {first_examples[0]['relevance_scores']}")
print(f"Example doc token IDs: {first_examples[0]['doc_token_ids']}")

In [ ]:
len(first_examples[0]['doc_token_ids'])

In [ ]:
# import json
# import random

# def create_first_dataset_full(raw_data_path, abstract_lookup, tokenizer, 
#                                max_candidates=100, samples_per_query=1):
#     """
#     Create FIRST training dataset using FULL candidate lists.
#     - max_candidates: upper limit to prevent OOM (set high like 100-200)
#     """
#     training_examples = []
    
#     # Pre-compute token IDs for all possible document identifiers
#     all_doc_ids = get_doc_ids(max_candidates)
#     doc_letter_to_token = {}
#     for letter in all_doc_ids:
#         tokens = tokenizer.encode(letter, add_special_tokens=False)
#         doc_letter_to_token[letter] = tokens[0]
    
#     with open(raw_data_path, 'r') as f:
#         for line in f:
#             example = json.loads(line)
#             query = example['query_text']
#             candidate_pool = example['candidate_pool']
#             candidate_scores = example['candidate_scores']
            
#             scored_candidates = list(zip(candidate_pool, candidate_scores))
            
#             # Need at least some relevant documents
#             has_relevant = any(s > 0 for _, s in scored_candidates)
#             if not has_relevant:
#                 continue
            
#             # Truncate if exceeds max_candidates (to prevent OOM)
#             if len(scored_candidates) > max_candidates:
#                 scored_candidates = scored_candidates[:max_candidates]
            
#             for _ in range(samples_per_query):
#                 # Use ALL candidates (just shuffle for position randomization)
#                 selected = scored_candidates.copy()
#                 random.shuffle(selected)
                
#                 # Build prompt and collect metadata
#                 doc_ids = get_doc_ids(len(selected))
#                 doc_ids_and_texts = []
#                 relevance_scores = []
#                 doc_token_ids = []
                
#                 for i, (cid, score) in enumerate(selected):
#                     doc_id = doc_ids[i]
#                     text = abstract_lookup.get(cid, f"Document {cid}")
#                     doc_ids_and_texts.append((doc_id, text))
#                     relevance_scores.append(float(score))
#                     doc_token_ids.append(doc_letter_to_token[doc_id])
                
#                 best_idx = relevance_scores.index(max(relevance_scores))
#                 target_id = doc_ids[best_idx]
                
#                 prompt = create_first_prompt(query, doc_ids_and_texts)
                
#                 training_examples.append({
#                     'prompt': prompt,
#                     'target': target_id,
#                     'query': query,
#                     'relevance_scores': relevance_scores,
#                     'doc_token_ids': doc_token_ids,
#                     'num_candidates': len(selected)
#                 })
    
#     return training_examples

# # Create dataset with full candidate lists
# first_examples = create_first_dataset_full(
#     "reranking_dataset_doris_mae.jsonl",
#     abstract_lookup,
#     tokenizer.tokenizer,
#     max_candidates=100,  # Match your inference max_k
#     samples_per_query=1
# )

# print(f"Created {len(first_examples)} FIRST training examples")
# print(f"Candidates per example: {first_examples[0]['num_candidates']}")

In [ ]:
# Check max token length across all examples
max_tokens = 0
token_lengths = []

for example in first_examples:
    messages = [
        {"role": "system", "content": FIRST_SYSTEM_PROMPT},
        {"role": "user", "content": example['prompt']},
    ]
    
    text = tokenizer.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    tokens = tokenizer.tokenizer(text, truncation=False, return_tensors=None)
    length = len(tokens['input_ids'])
    token_lengths.append(length)
    
    if length > max_tokens:
        max_tokens = length

print(f"Max tokens: {max_tokens}")
print(f"Mean tokens: {sum(token_lengths) / len(token_lengths):.0f}")
print(f"Min tokens: {min(token_lengths)}")

# Check how many exceed your max_length
exceeds_limit = sum(1 for l in token_lengths if l > 16384)
print(f"Examples exceeding 16384 tokens: {exceeds_limit} ({100*exceeds_limit/len(token_lengths):.1f}%)")

In [ ]:
from datasets import Dataset

# Get the underlying text tokenizer from the processor
text_tokenizer = tokenizer.tokenizer  # This is the actual text tokenizer

def format_for_first_training(example):
    """Format example with input_ids, attention_mask, and FIRST metadata."""
    messages = [
        {"role": "system", "content": FIRST_SYSTEM_PROMPT},
        {"role": "user", "content": example['prompt']},
    ]
    
    # Use the text tokenizer's apply_chat_template
    text = text_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize using the text tokenizer
    encodings = text_tokenizer(
        text,
        truncation=True,
        max_length=16384,
        padding=False,
        return_tensors=None
    )
    
    return {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "relevance_scores": example['relevance_scores'],
        "doc_token_ids": example['doc_token_ids'],
    }

# Create HuggingFace Dataset
first_dataset = Dataset.from_list(first_examples)
first_dataset = first_dataset.map(format_for_first_training, remove_columns=['prompt', 'target', 'query', 'num_candidates'])

print(first_dataset)
print(f"\nFeatures: {first_dataset.features}")

In [ ]:
for i in get_doc_ids(50):
    print(i, tokenizer.tokenizer.tokenize(i))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class FIRSTLoss(nn.Module):
    def __init__(self, temperature=1.0, use_gains=True):
        super().__init__()
        self.temperature = temperature
        self.use_gains = use_gains
    
    def forward(self, logits, attention_mask, doc_token_ids, relevance_scores, num_docs):
        batch_size = logits.shape[0]
        total_loss = 0.0
        
        for b in range(batch_size):
            seq_len = attention_mask[b].sum().item()
            seq_logits = logits[b, int(seq_len) - 1, :]
            
            n_docs = int(num_docs[b].item())
            if n_docs == 0:
                continue
                
            doc_logits = seq_logits[doc_token_ids[b, :n_docs]]
            doc_logits = doc_logits / self.temperature
            
            rel_scores = relevance_scores[b, :n_docs]
            ideal_ranking = torch.argsort(rel_scores, descending=True)
            
            # Compute NDCG-style gains
            if self.use_gains:
                gains = (2.0 ** rel_scores - 1.0)
            else:
                gains = torch.ones_like(rel_scores)
            
            # ListMLE loss
            listmle_loss = 0.0
            n = len(ideal_ranking)
            
            for i in range(n - 1):
                pos = ideal_ranking[i]
                remaining_positions = ideal_ranking[i:]
                remaining_logits = doc_logits[remaining_positions]
                
                gain_weight = gains[pos]
                listmle_loss += gain_weight * (
                    torch.logsumexp(remaining_logits, dim=0) - doc_logits[pos]
                )
            
            # Normalize by total gain to keep loss magnitude stable
            total_gain = gains.sum()
            if total_gain > 1e-8:
                listmle_loss = listmle_loss / total_gain
            else:
                listmle_loss = listmle_loss / max(n - 1, 1)
            
            total_loss += listmle_loss
        
        return total_loss / batch_size


class FIRSTTrainer(Trainer):
    """Custom trainer for FIRST with listwise ranking loss."""
    
    def __init__(self, *args, doc_token_ids_field="doc_token_ids", 
                 relevance_scores_field="relevance_scores", 
                 loss_temperature=1.0, 
                 use_gains=True,
                 **kwargs):
        super().__init__(*args, **kwargs)
        self.doc_token_ids_field = doc_token_ids_field
        self.relevance_scores_field = relevance_scores_field
        self.first_loss = FIRSTLoss(temperature=loss_temperature, use_gains=use_gains)
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        doc_token_ids = inputs.pop(self.doc_token_ids_field)
        relevance_scores = inputs.pop(self.relevance_scores_field)
        num_docs = inputs.pop("num_docs")
        
        outputs = model(**inputs)
        logits = outputs.logits
        
        loss = self.first_loss(
            logits=logits,
            attention_mask=inputs["attention_mask"],
            doc_token_ids=doc_token_ids,
            relevance_scores=relevance_scores,
            num_docs=num_docs
        )
        
        return (loss, outputs) if return_outputs else loss

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Any
import torch

@dataclass
class FIRSTDataCollator:
    """Collator that handles variable-length sequences and FIRST metadata."""
    tokenizer: Any  # This will be the Gemma3Processor
    padding: bool = True
    max_length: int = 16384
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Separate FIRST-specific fields
        relevance_scores = [f.pop('relevance_scores') for f in features]
        doc_token_ids = [f.pop('doc_token_ids') for f in features]
        
        # Get the underlying text tokenizer for padding
        text_tokenizer = self.tokenizer.tokenizer if hasattr(self.tokenizer, 'tokenizer') else self.tokenizer
        
        # Pad sequences using the text tokenizer
        batch = text_tokenizer.pad(
            features,
            padding=self.padding,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        # Convert FIRST fields to tensors
        max_docs = max(len(r) for r in relevance_scores)
        num_docs_list = torch.tensor([len(r) for r in relevance_scores], dtype=torch.long)

        # Pad relevance scores and doc_token_ids
        padded_relevance = torch.zeros(len(relevance_scores), max_docs)
        padded_doc_tokens = torch.zeros(len(doc_token_ids), max_docs, dtype=torch.long)
        
        for i, (rel, doc) in enumerate(zip(relevance_scores, doc_token_ids)):
            padded_relevance[i, :len(rel)] = torch.tensor(rel)
            padded_doc_tokens[i, :len(doc)] = torch.tensor(doc)
        
        batch['relevance_scores'] = padded_relevance
        batch['doc_token_ids'] = padded_doc_tokens
        batch['num_docs'] = num_docs_list

        return batch

In [ ]:
# from transformers import TrainingArguments

# small_dataset = first_dataset.select(range(min(100, len(first_dataset))))
# data_collator = FIRSTDataCollator(tokenizer=tokenizer, max_length=16384)

# test_trainer = FIRSTTrainer(
#     model=model,
#     args=TrainingArguments(
#         output_dir="test-run",
#         per_device_train_batch_size=1,
#         max_steps=20,
#         logging_steps=5,
#         report_to="none",
#         remove_unused_columns=False,
#     ),
#     train_dataset=small_dataset,
#     data_collator=data_collator,
#     loss_temperature=1.0,
# )
# test_trainer.train()

In [ ]:
from transformers import TrainingArguments
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

training_args = TrainingArguments(
    output_dir=f"gemma-3-12b-first-reranker-{timestamp}",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    num_train_epochs=2,
    warmup_steps = 20,
    warmup_ratio=0.06,
    logging_steps=1,
    # save_steps=500,
    save_total_limit=3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    optim="adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
    remove_unused_columns=False,  # Important! Keep our custom columns
)

data_collator = FIRSTDataCollator(tokenizer=tokenizer, max_length=16384)

trainer = FIRSTTrainer(
    model=model,
    args=training_args,
    train_dataset=first_dataset,
    data_collator=data_collator,
    loss_temperature=1.0,
    use_gains=True
)

trainer.train()

In [ ]:
model.save_pretrained(f"gemma-3-12b-first-reranker-{timestamp}")
tokenizer.save_pretrained(f"gemma-3-12b-first-reranker-{timestamp}")

# Or save merged model for GGUF conversion
model.save_pretrained_merged(
    f"gemma-3-12b-first-reranker-merged-{timestamp}",
    tokenizer,
    save_method="merged_16bit",
)